## Agent的流式输出
通过 `invoke` 调用Agent时，内部可能经历多次调用，长时间看不到调用情况，用户体验不好，可以通过 `流式调用` （渐进式显示输出）优化用户体验， `实时显示` Agent 运行过程中的更新。特别是在处理LLM 延迟时尤其有效。

通过“ agent.stream(stream_mode=指定模式) ”来指定。具体模式有：values、updates（默认）、messages、custom、checkpoints、tasks、debug。

### 1. 具体的输出模式

1.1  values输出模式(每一步都要获取完整状态)

In [6]:
import os
from idlelib import config
from typing import Dict, Any

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from rich import print as rprint

load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID,用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三", "level": "VIP", "join_date": "2022-12-02"}

def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID,用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费数等
    """
    return {"total_orders": 14, "total_spent": 287192.83}

@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-07-07"
    }

customer_service_agent = create_agent(
    model = model_openai,
    tools=[
        query_customer_data,
        check_order_history,
        get_current_promotions,
    ]
)

for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user", "content": "查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠"}]},
        stream_mode="values"
):
    rprint(chunk)
    print("-" * 60)


{
    'messages': [
        HumanMessage(
            content='查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠',
            additional_kwargs={},
            response_metadata={},
            id='575a391b-3a7d-42f2-93cb-2a2d04592368'
        )
    ]
}

------------------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠',
            additional_kwargs={},
            response_metadata={},
            id='575a391b-3a7d-42f2-93cb-2a2d04592368'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 982,
                    'prompt_tokens': 488,
                    'total_tokens': 1470,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 889,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 982
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'text_tokens': 488
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-e6f16eaa-d56a-9c68-8621-414a4c2330c2',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0c379-cd09-7451-b9a3-a717fbeb1049-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'CUSTID1234567'},
                    'id': 'call_8e80a61fab4d4c918fc8d290',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'CUSTID1234567'},
                    'id': 'call_fbeffdf3ba4143bbaed823fa',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_4cb8a3a373654fd2a7d748b1',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 488,
                'output_tokens': 982,
                'total_tokens': 1470,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 889}
            }
        )
    ]
}

------------------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠',
            additional_kwargs={},
            response_metadata={},
            id='575a391b-3a7d-42f2-93cb-2a2d04592368'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 982,
                    'prompt_tokens': 488,
                    'total_tokens': 1470,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 889,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 982
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'text_tokens': 488
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-e6f16eaa-d56a-9c68-8621-414a4c2330c2',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0c379-cd09-7451-b9a3-a717fbeb1049-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'CUSTID1234567'},
                    'id': 'call_8e80a61fab4d4c918fc8d290',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'CUSTID1234567'},
                    'id': 'call_fbeffdf3ba4143bbaed823fa',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_4cb8a3a373654fd2a7d748b1',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 488,
                'output_tokens': 982,
                'total_tokens': 1470,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 889}
            }
        ),
        ToolMessage(
            content='{"name": "张三", "level": "VIP", "join_date": "2022-12-02"}',
            name='query_customer_data',
            id='abae01a2-b16d-4647-b04c-5944e31d20a9',
            tool_call_id='call_8e80a61fab4d4c918fc8d290'
        ),
        ToolMessage(
            content='{"total_orders": 14, "total_spent": 287192.83}',
            name='check_order_history',
            id='ef473a43-f803-489f-8abe-fefb71a17cf6',
            tool_call_id='call_fbeffdf3ba4143bbaed823fa'
        ),
        ToolMessage(
            content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-07-07"}',
            name='get_current_promotions',
            id='dba4b5b6-ea29-48c3-9087-13378a56ba6a',
            tool_call_id='call_4cb8a3a373654fd2a7d748b1'
        )
    ]
}

------------------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠',
            additional_kwargs={},
            response_metadata={},
            id='575a391b-3a7d-42f2-93cb-2a2d04592368'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 982,
                    'prompt_tokens': 488,
                    'total_tokens': 1470,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 889,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 982
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'text_tokens': 488
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-e6f16eaa-d56a-9c68-8621-414a4c2330c2',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0c379-cd09-7451-b9a3-a717fbeb1049-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'CUSTID1234567'},
                    'id': 'call_8e80a61fab4d4c918fc8d290',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'CUSTID1234567'},
                    'id': 'call_fbeffdf3ba4143bbaed823fa',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_4cb8a3a373654fd2a7d748b1',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 488,
                'output_tokens': 982,
                'total_tokens': 1470,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 889}
            }
        ),
        ToolMessage(
            content='{"name": "张三", "level": "VIP", "join_date": "2022-12-02"}',
            name='query_customer_data',
            id='abae01a2-b16d-4647-b04c-5944e31d20a9',
            tool_call_id='call_8e80a61fab4d4c918fc8d290'
        ),
        ToolMessage(
            content='{"total_orders": 14, "total_spent": 287192.83}',
            name='check_order_history',
            id='ef473a43-f803-489f-8abe-fefb71a17cf6',
            tool_call_id='call_fbeffdf3ba4143bbaed823fa'
        ),
        ToolMessage(
            content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-07-07"}',
            name='get_current_promotions',
            id='dba4b5b6-ea29-48c3-9087-13378a56ba6a',
            tool_call_id='call_4cb8a3a373654fd2a7d748b1'
        ),
        AIMessage(
            content='已为您查询到客户 ID **CUSTID1234567** 的相关信息，具体如下：\n\n### 👤 个人信息\n- 
**姓名**：张三\n- **会员等级**：VIP\n- **加入日期**：2022年12月2日\n\n### 📦 历史订单\n- **总订单数**：14 单\n- 
**累计消费金额**：¥287,192.83\n\n### 🎁 当前可用促销优惠\n- **优惠活动**：老用户优惠、会员专属折扣\n- 
**有效期至**：2027年7月7日\n\n如有其他需求或需进一步查询某项详细信息，请随时告知！',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 326,
                    'prompt_tokens': 680,
                    'total_tokens': 1006,
                    'completion_tokens_details': {
                        'accepted_pre

------------------------------------------------------------


1.2  updates输出模式(默认模式， 只增量更新状态中发生变化的内容)

In [7]:
from typing import Dict, Any
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
import os

load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID,用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三", "level": "VIP", "join_date": "2022-12-02"}

def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID,用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费数等
    """
    return {"total_orders": 14, "total_spent": 287192.83}

@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-07-07"
    }

customer_service_agent = create_agent(
    model = model_openai,
    tools=[
        query_customer_data,
        check_order_history,
        get_current_promotions,
    ]
)

for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user", "content": "查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠"}]},
        stream_mode="updates"
):
    rprint(chunk)
    print("-" * 60)

{
    'model': {
        'messages': [
            AIMessage(
                content='',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 616,
                        'prompt_tokens': 488,
                        'total_tokens': 1104,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 523,
                            'rejected_prediction_tokens': None,
                            'text_tokens': 616
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'text_tokens': 488
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'qwen3.7-flash',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-64197bf0-e269-93e4-9d96-89ec553d7ac4',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a0c37c-6715-7480-be8d-3932c73d1d0c-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'CUSTID1234567'},
                        'id': 'call_e85fcd0d5aa74f2ea3e55ef3',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'CUSTID1234567'},
                        'id': 'call_fa122955bf4143bbab9235bd',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_4b2e2ee05d414816bb317b91',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 488,
                    'output_tokens': 616,
                    'total_tokens': 1104,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 523}
                }
            )
        ]
    }
}

------------------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"total_orders": 14, "total_spent": 287192.83}',
                name='check_order_history',
                id='ad0bfae5-cd36-4381-ab7e-89511f732ab0',
                tool_call_id='call_fa122955bf4143bbab9235bd'
            )
        ]
    }
}

------------------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2022-12-02"}',
                name='query_customer_data',
                id='a7326cd2-5d21-4546-9a4d-1e102faf6c0f',
                tool_call_id='call_e85fcd0d5aa74f2ea3e55ef3'
            )
        ]
    }
}

------------------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-07-07"}',
                name='get_current_promotions',
                id='c10c46ed-08a6-4c98-aae4-f061b5dc497b',
                tool_call_id='call_4b2e2ee05d414816bb317b91'
            )
        ]
    }
}

------------------------------------------------------------


{
    'model': {
        'messages': [
            AIMessage(
                content='您好！为您查询到客户 ID 为 `CUSTID1234567` 的相关详细信息如下：\n\n**👤 个人基本信息**\n* 
**姓名**：张三\n* **客户等级**：VIP\n* **加入日期**：2022-12-02\n\n**📦 历史订单情况**\n* **总订单数**：14 笔\n* 
**累计消费金额**：¥287,192.83\n\n**🎁 当前可用优惠活动**\n* **活动名称**：老用户优惠、会员专属折扣\n* 
**有效截止日期**：2027-07-07\n\n如有其他需求，请随时告知！',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 271,
                        'prompt_tokens': 680,
                        'total_tokens': 951,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 110,
                            'rejected_prediction_tokens': None,
                            'text_tokens': 271
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'text_tokens': 680
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'qwen3.7-flash',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-3fcf1032-d63d-9df9-8928-057971146938',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a0c37c-8956-75c3-a373-61b47f184075-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 680,
                    'output_tokens': 271,
                    'total_tokens': 951,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 110}
                }
            )
        ]
    }
}

------------------------------------------------------------


1.3 messages输出模式

该模式中会输出流式返回的Token以及相关的元数据（如：来自哪个节点），可以用在实现类似ChatGPT 的打字机效果场景，为聊天机器人等交互式应用提供最佳的实时体验。

In [10]:
from typing import Dict, Any
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID,用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三", "level": "VIP", "join_date": "2022-12-02"}

def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID,用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费数等
    """
    return {"total_orders": 14, "total_spent": 287192.83}

@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-07-07"
    }

customer_service_agent = create_agent(
    model = model_openai,
    tools=[
        query_customer_data,
        check_order_history,
        get_current_promotions,
    ]
)

for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user", "content": "查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠"}]},
        stream_mode="messages"
):
    print(chunk[0].content,end="",flush=True)

{"total_orders": 14, "total_spent": 287192.83}{"name": "张三", "level": "VIP", "join_date": "2022-12-02"}{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-07-07"}为您查询到客户 ID 为 `CUSTID1234567` 的相关信息如下：

**👤 客户基本信息**
*   **姓名**：张三
*   **会员等级**：VIP
*   **加入日期**：2022-12-02

**📦 历史订单记录**
*   **总订单数**：14 单
*   **累计消费金额**：¥287,192.83

**🎁 当前可用优惠活动**
*   **活动名称**：老用户优惠、会员专属折扣
*   **活动有效期至**：2027-07-07

如需查看更详细的订单明细或了解具体活动规则，请随时告诉我！

1.4 tasks输出模式

该模式会输出当前task任务开始和结束的时间，包含任务的结果和错误信息，该模式用于监控任务的生命周期。

In [13]:
from typing import Dict, Any
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
import os

load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID,用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三", "level": "VIP", "join_date": "2022-12-02"}

def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID,用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费数等
    """
    return {"total_orders": 14, "total_spent": 287192.83}

@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-07-07"
    }

customer_service_agent = create_agent(
    model = model_openai,
    tools=[
        query_customer_data,
        check_order_history,
        get_current_promotions,
    ]
)

for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user", "content": "查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠"}]},
        stream_mode="tasks"
):
    rprint(chunk)
    print("-" * 70)
   # print(chunk[0].content,end="",flush=True)

{
    'id': '7000d651-8fa1-55eb-ea6f-b95723a2301c',
    'name': 'model',
    'input': {
        'messages': [
            HumanMessage(
                content='查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠',
                additional_kwargs={},
                response_metadata={},
                id='b03de92d-74d4-4319-b1a5-bbd99a0dd5f2'
            )
        ]
    },
    'triggers': ('branch:to:model',),
    'metadata': {'ls_integration': 'langchain_create_agent'}
}

----------------------------------------------------------------------


{
    'id': '7000d651-8fa1-55eb-ea6f-b95723a2301c',
    'name': 'model',
    'error': None,
    'result': {
        'messages': [
            AIMessage(
                content='',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 1288,
                        'prompt_tokens': 488,
                        'total_tokens': 1776,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 1195,
                            'rejected_prediction_tokens': None,
                            'text_tokens': 1288
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'text_tokens': 488
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'qwen3.7-flash',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-4ce77574-3c43-9874-8f4c-d6c4c4b65cf2',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a0c387-4405-7842-ab86-71709a9b6b48-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'CUSTID1234567'},
                        'id': 'call_0c1dbee2e5a847eba4b817c0',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'CUSTID1234567'},
                        'id': 'call_b3133ce93b01433c9ee0f0d5',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_723575b73d4f4498b60ea09a',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 488,
                    'output_tokens': 1288,
                    'total_tokens': 1776,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 1195}
                }
            )
        ]
    },
    'interrupts': []
}

----------------------------------------------------------------------


{
    'id': '8dee2025-2552-e268-3426-6a6c89be7513',
    'name': 'tools',
    'input': [
        {
            'name': 'query_customer_data',
            'args': {'customer_id': 'CUSTID1234567'},
            'id': 'call_0c1dbee2e5a847eba4b817c0',
            'type': 'tool_call'
        }
    ],
    'triggers': ('__pregel_push',),
    'metadata': {'ls_integration': 'langchain_create_agent'}
}

...

----------------------------------------------------------------------


{
    'id': '0428d938-fe40-04c5-aa89-0c3a5f426eca',
    'name': 'model',
    'input': {
        'messages': [
            HumanMessage(
                content='查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠',
                additional_kwargs={},
                response_metadata={},
                id='b03de92d-74d4-4319-b1a5-bbd99a0dd5f2'
            ),
            AIMessage(
                content='',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 1288,
                        'prompt_tokens': 488,
                        'total_tokens': 1776,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 1195,
                            'rejected_prediction_tokens': None,
                            'text_tokens': 1288
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'text_tokens': 488
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'qwen3.7-flash',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-4ce77574-3c43-9874-8f4c-d6c4c4b65cf2',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a0c387-4405-7842-ab86-71709a9b6b48-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'CUSTID1234567'},
                        'id': 'call_0c1dbee2e5a847eba4b817c0',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'CUSTID1234567'},
                        'id': 'call_b3133ce93b01433c9ee0f0d5',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_723575b73d4f4498b60ea09a',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 488,
                    'output_tokens': 1288,
                    'total_tokens': 1776,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 1195}
                }
            ),
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2022-12-02"}',
                name='query_customer_data',
                id='c887ea68-c7d5-458a-926d-a596b6b48dbd',
                tool_call_id='call_0c1dbee2e5a847eba4b817c0'
            ),
            ToolMessage(
                content='{"total_orders": 14, "total_spent": 287192.83}',
                name='check_order_history',
                id='522ac1da-d345-4d69-9768-3ebfb04cffe2',
                tool_call_id='call_b3133ce93b01433c9ee0f0d5'
            ),
            ToolMessage(
                content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-07-07"}',
                name='get_current_promotions',
                id='70d7cf9f-8169-4993-8257-d9369530daca',
                tool_call_id='call_723575b73d4f4498b60ea09a'
            )
        ]
    },
    'triggers': ('branch:to:model',),
    'metadata': {'ls_integration': 'langchain_create_agent'}
}

----------------------------------------------------------------------


{
    'id': '0428d938-fe40-04c5-aa89-0c3a5f426eca',
    'name': 'model',
    'error': None,
    'result': {
        'messages': [
            AIMessage(
                content='您好，客户ID为 CUSTID1234567 的相关信息如下：\n\n**👤 个人信息**\n*   **姓名**：张三\n*   
**等级**：VIP\n*   **加入日期**：2022年12月2日\n\n**📦 历史订单**\n*   **总订单数**：14单\n*   
**累计消费金额**：287,192.83元\n\n**🎁 当前可用优惠**\n*   **促销活动**：老用户优惠、会员专属折扣\n*   
**有效期至**：2027年7月7日',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 312,
                        'prompt_tokens': 680,
                        'total_tokens': 992,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 167,
                            'rejected_prediction_tokens': None,
                            'text_tokens': 312
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'text_tokens': 680
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'qwen3.7-flash',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-4501d468-9239-9548-98ab-a5ecc48c3fe5',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a0c387-8af2-7711-b7a8-c1843b373f70-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 680,
                    'output_tokens': 312,
                    'total_tokens': 992,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 167}
                }
            )
        ]
    },
    'interrupts': []
}

----------------------------------------------------------------------


1.5 debug输出模式

该模式与tasks模式类似，比task模式多输出任务步骤、时间戳、task类型（task/task_result），该模式用于调试、监控task任务的生命周期。

In [14]:
from typing import Dict, Any
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
import os

load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID,用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三", "level": "VIP", "join_date": "2022-12-02"}

def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID,用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费数等
    """
    return {"total_orders": 14, "total_spent": 287192.83}

@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-07-07"
    }

customer_service_agent = create_agent(
    model = model_openai,
    tools=[
        query_customer_data,
        check_order_history,
        get_current_promotions,
    ]
)

for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user", "content": "查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠"}]},
        stream_mode="debug"
):
    rprint(chunk)
    print("-" * 70)
   # print(chunk[0].content,end="",flush=True)

{
    'step': 1,
    'timestamp': '2026-09-21T10:36:09.943371+00:00',
    'type': 'task',
    'payload': {
        'id': '18c46cbb-8c85-a3c3-c68e-335254ea4421',
        'name': 'model',
        'input': {
            'messages': [
                HumanMessage(
                    content='查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠',
                    additional_kwargs={},
                    response_metadata={},
                    id='d8df778c-9348-4138-8fca-78988c90b176'
                )
            ]
        },
        'triggers': ('branch:to:model',),
        'metadata': {'ls_integration': 'langchain_create_agent'}
    }
}

----------------------------------------------------------------------


{
    'step': 1,
    'timestamp': '2026-09-21T10:36:16.090654+00:00',
    'type': 'task_result',
    'payload': {
        'id': '18c46cbb-8c85-a3c3-c68e-335254ea4421',
        'name': 'model',
        'error': None,
        'result': {
            'messages': [
                AIMessage(
                    content='',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 437,
                            'prompt_tokens': 488,
                            'total_tokens': 925,
                            'completion_tokens_details': {
                                'accepted_prediction_tokens': None,
                                'audio_tokens': None,
                                'reasoning_tokens': 344,
                                'rejected_prediction_tokens': None,
                                'text_tokens': 437
                            },
                            'prompt_tokens_details': {
                                'audio_tokens': None,
                                'cache_write_tokens': None,
                                'cached_tokens': 0,
                                'text_tokens': 488
                            }
                        },
                        'model_provider': 'openai',
                        'model_name': 'qwen3.7-flash',
                        'system_fingerprint': None,
                        'id': 'chatcmpl-eb0f2d33-752c-9509-954b-9cc6a017d349',
                        'finish_reason': 'tool_calls',
                        'logprobs': None
                    },
                    id='lc_run--01a0c389-995d-7cf0-ad59-5d65b52d26e1-0',
                    tool_calls=[
                        {
                            'name': 'query_customer_data',
                            'args': {'customer_id': 'CUSTID1234567'},
                            'id': 'call_b352dd897e5342bca08d8b5e',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'check_order_history',
                            'args': {'customer_id': 'CUSTID1234567'},
                            'id': 'call_36856ffa929449c086ebce81',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'get_current_promotions',
                            'args': {},
                            'id': 'call_6cc5168b27f34988bdf8e867',
                            'type': 'tool_call'
                        }
                    ],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 488,
                        'output_tokens': 437,
                        'total_tokens': 925,
                        'input_token_details': {'cache_read': 0},
                        'output_token_details': {'reasoning': 344}
                    }
                )
            ]
        },
        'interrupts': []
    }
}

----------------------------------------------------------------------


{
    'step': 2,
    'timestamp': '2026-09-21T10:36:16.090654+00:00',
    'type': 'task',
    'payload': {
        'id': 'a67f0e16-cb9f-5172-090f-c0e0ab4c3798',
        'name': 'tools',
        'input': [
            {
                'name': 'query_customer_data',
                'args': {'customer_id': 'CUSTID1234567'},
                'id': 'call_b352dd897e5342bca08d8b5e',
                'type': 'tool_call'
            }
        ],
        'triggers': ('__pregel_push',),
        'metadata': {'ls_integration': 'langchain_create_agent'}
    }
}

...

----------------------------------------------------------------------


{
    'step': 3,
    'timestamp': '2026-09-21T10:36:16.126531+00:00',
    'type': 'task',
    'payload': {
        'id': '18620bbf-622b-3131-ba59-3c06b525915c',
        'name': 'model',
        'input': {
            'messages': [
                HumanMessage(
                    content='查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠',
                    additional_kwargs={},
                    response_metadata={},
                    id='d8df778c-9348-4138-8fca-78988c90b176'
                ),
                AIMessage(
                    content='',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 437,
                            'prompt_tokens': 488,
                            'total_tokens': 925,
                            'completion_tokens_details': {
                                'accepted_prediction_tokens': None,
                                'audio_tokens': None,
                                'reasoning_tokens': 344,
                                'rejected_prediction_tokens': None,
                                'text_tokens': 437
                            },
                            'prompt_tokens_details': {
                                'audio_tokens': None,
                                'cache_write_tokens': None,
                                'cached_tokens': 0,
                                'text_tokens': 488
                            }
                        },
                        'model_provider': 'openai',
                        'model_name': 'qwen3.7-flash',
                        'system_fingerprint': None,
                        'id': 'chatcmpl-eb0f2d33-752c-9509-954b-9cc6a017d349',
                        'finish_reason': 'tool_calls',
                        'logprobs': None
                    },
                    id='lc_run--01a0c389-995d-7cf0-ad59-5d65b52d26e1-0',
                    tool_calls=[
                        {
                            'name': 'query_customer_data',
                            'args': {'customer_id': 'CUSTID1234567'},
                            'id': 'call_b352dd897e5342bca08d8b5e',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'check_order_history',
                            'args': {'customer_id': 'CUSTID1234567'},
                            'id': 'call_36856ffa929449c086ebce81',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'get_current_promotions',
                            'args': {},
                            'id': 'call_6cc5168b27f34988bdf8e867',
                            'type': 'tool_call'
                        }
                    ],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 488,
                        'output_tokens': 437,
                        'total_tokens': 925,
                        'input_token_details': {'cache_read': 0},
                        'output_token_details': {'reasoning': 344}
                    }
                ),
                ToolMessage(
                    content='{"name": "张三", "level": "VIP", "join_date": "2022-12-02"}',
                    name='query_customer_data',
                    id='030effdb-eba0-4e0d-a1dd-1762f4e61217',
                    tool_call_id='call_b352dd897e5342bca08d8b5e'
                ),
                ToolMessage(
                    content='{"total_orders": 14, "total_spent": 287192.83}',
                    name='check_order_history',
                    id='9ec3a180-e77a-4edf-96ae-3c8bc71cf0aa',
                    tool_call_id='call_36856ffa929449c086ebce81'
                ),
                ToolMessage(
                    content='{"promotions": ["老用户优惠", "会员专

----------------------------------------------------------------------


{
    'step': 3,
    'timestamp': '2026-09-21T10:36:20.610092+00:00',
    'type': 'task_result',
    'payload': {
        'id': '18620bbf-622b-3131-ba59-3c06b525915c',
        'name': 'model',
        'error': None,
        'result': {
            'messages': [
                AIMessage(
                    content='为您查询到客户 ID **CUSTID1234567** 的相关信息如下：\n\n**👤 个人信息**\n*   
**姓名**：张三\n*   **会员等级**：VIP\n*   **加入日期**：2022年12月2日\n\n**📦 历史订单**\n*   **总订单数**：14 
单\n*   **累计消费金额**：¥287,192.83\n\n**🎁 当前可用优惠活动**\n*   **活动名称**：老用户优惠、会员专属折扣\n*   
**有效期限**：至 2027年7月7日\n\n如有其他需求，请随时告知！',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 291,
                            'prompt_tokens': 680,
                            'total_tokens': 971,
                            'completion_tokens_details': {
                                'accepted_prediction_tokens': None,
                                'audio_tokens': None,
                                'reasoning_tokens': 129,
                                'rejected_prediction_tokens': None,
                                'text_tokens': 291
                            },
                            'prompt_tokens_details': {
                                'audio_tokens': None,
                                'cache_write_tokens': None,
                                'cached_tokens': 0,
                                'text_tokens': 680
                            }
                        },
                        'model_provider': 'openai',
                        'model_name': 'qwen3.7-flash',
                        'system_fingerprint': None,
                        'id': 'chatcmpl-1b886423-1ac2-9470-b10b-29a0148e2278',
                        'finish_reason': 'stop',
                        'logprobs': None
                    },
                    id='lc_run--01a0c389-b18c-77c3-84c2-af43f176bbc2-0',
                    tool_calls=[],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 680,
                        'output_tokens': 291,
                        'total_tokens': 971,
                        'input_token_details': {'cache_read': 0},
                        'output_token_details': {'reasoning': 129}
                    }
                )
            ]
        },
        'interrupts': []
    }
}

----------------------------------------------------------------------


1.6 checkpoints输出模式

该模式中，每当检查点（checkpoint）被创建时会触发输出，输出包含检查点中的状态，用于需要状态持久化、工作流恢复或分布式执行跟踪的高级场景。

In [17]:
from typing import Dict, Any
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from rich import print as rprint
import os

load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID,用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三", "level": "VIP", "join_date": "2022-12-02"}

def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID,用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费数等
    """
    return {"total_orders": 14, "total_spent": 287192.83}

@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-07-07"
    }

# 1. 创建内存检查点存储
check_pointer = InMemorySaver()

# 2. 创建Agent
customer_service_agent = create_agent(
    model = model_openai,
    tools=[
        query_customer_data,
        check_order_history,
        get_current_promotions,
    ],
    checkpointer=check_pointer, #启用检查点
)

# 3. 创建唯一的会话ID
custom_config = {"configurable": {"thread_id": "session01"}}

# 4.调用Agent
checkpoint_count = 0

# 使用checkpointer模式进行流式监控
for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user", "content": "查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠"}]},
        config=custom_config,
        stream_mode="checkpoints"
):
    checkpoint_count += 1
    print(f"检查点 #{checkpoint_count}")
    rprint(chunk)
    print("-" * 70)
   # print(chunk[0].content,end="",flush=True)

检查点 #1


{
    'config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1b5b90-ed8f-6d91-bfff-91ea280ba02e'
        }
    },
    'parent_config': None,
    'values': {'messages': []},
    'metadata': {'source': 'input', 'step': -1, 'parents': {}},
    'next': ['__start__'],
    'tasks': [
        {'id': '71e3916f-d8e7-4582-a2da-795a313e498a', 'name': '__start__', 'interrupts': (), 'state': None}
    ]
}

----------------------------------------------------------------------
检查点 #2


{
    'config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1b5b90-ed99-6963-8000-c6e113b34b5d'
        }
    },
    'parent_config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1b5b90-ed8f-6d91-bfff-91ea280ba02e'
        }
    },
    'values': {
        'messages': [
            HumanMessage(
                content='查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠',
                additional_kwargs={},
                response_metadata={},
                id='12cb5920-5311-4674-a26e-053e872222d4'
            )
        ]
    },
    'metadata': {'source': 'loop', 'step': 0, 'parents': {}},
    'next': ['model'],
    'tasks': [{'id': '72d7cf91-fe32-7571-3c6c-8a3c6753aa2b', 'name': 'model', 'interrupts': (), 'state': None}]
}

----------------------------------------------------------------------
检查点 #3


...

----------------------------------------------------------------------
检查点 #4


{
    'config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1b5b91-3c06-6954-8002-2a07f14f4752'
        }
    },
    'parent_config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1b5b91-3bd8-6665-8001-7c095d0c84c2'
        }
    },
    'values': {
        'messages': [
            HumanMessage(
                content='查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠',
                additional_kwargs={},
                response_metadata={},
                id='12cb5920-5311-4674-a26e-053e872222d4'
            ),
            AIMessage(
                content='',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 602,
                        'prompt_tokens': 488,
                        'total_tokens': 1090,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 509,
                            'rejected_prediction_tokens': None,
                            'text_tokens': 602
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'text_tokens': 488
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'qwen3.7-flash',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-2da2087f-a4fc-968a-9dc3-45a93ae594c6',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a0c3f7-b05f-7160-a834-1a4d4a5cc26e-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'CUSTID1234567'},
                        'id': 'call_eed775d548b4469c8394f91d',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'CUSTID1234567'},
                        'id': 'call_0831a365489049a1a1ec3ea3',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_710d87970d154f0da614f9f9',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 488,
                    'output_tokens': 602,
                    'total_tokens': 1090,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 509}
                }
            ),
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2022-12-02"}',
                name='query_customer_data',
                id='638ab20e-c84f-4b48-a990-f133595580ad',
                tool_call_id='call_eed775d548b4469c8394f91d'
            ),
            ToolMessage(
                content='{"total_orders": 14, "total_spent": 287192.83}',
                name='check_order_history',
                id='28b9f5a4-e097-4b46-a9f4-0d21eae5ae71',
                tool_call_id='call_0831a365489049a1a1ec3ea3'
            ),
            ToolMessage(
                content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-07-07"}',
                name='get_current_promotions',
                i

----------------------------------------------------------------------
检查点 #5


{
    'config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1b5b91-8165-6b26-8003-394b2e5f61c6'
        }
    },
    'parent_config': {
        'configurable': {
            'checkpoint_ns': '',
            'thread_id': 'session01',
            'checkpoint_id': '1f1b5b91-3c06-6954-8002-2a07f14f4752'
        }
    },
    'values': {
        'messages': [
            HumanMessage(
                content='查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠',
                additional_kwargs={},
                response_metadata={},
                id='12cb5920-5311-4674-a26e-053e872222d4'
            ),
            AIMessage(
                content='',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 602,
                        'prompt_tokens': 488,
                        'total_tokens': 1090,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 509,
                            'rejected_prediction_tokens': None,
                            'text_tokens': 602
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'text_tokens': 488
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'qwen3.7-flash',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-2da2087f-a4fc-968a-9dc3-45a93ae594c6',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a0c3f7-b05f-7160-a834-1a4d4a5cc26e-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'CUSTID1234567'},
                        'id': 'call_eed775d548b4469c8394f91d',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'CUSTID1234567'},
                        'id': 'call_0831a365489049a1a1ec3ea3',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_710d87970d154f0da614f9f9',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 488,
                    'output_tokens': 602,
                    'total_tokens': 1090,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 509}
                }
            ),
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2022-12-02"}',
                name='query_customer_data',
                id='638ab20e-c84f-4b48-a990-f133595580ad',
                tool_call_id='call_eed775d548b4469c8394f91d'
            ),
            ToolMessage(
                content='{"total_orders": 14, "total_spent": 287192.83}',
                name='check_order_history',
                id='28b9f5a4-e097-4b46-a9f4-0d21eae5ae71',
                tool_call_id='call_0831a365489049a1a1ec3ea3'
            ),
            ToolMessage(
                content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-07-07"}',
                name='get_current_promotions',
                i

----------------------------------------------------------------------


1.7 custom输出模式

开发者通过 get_stream_writer 在工具或节点内部 自定义发送的数据 ，用于 输出 业务逻辑相关的进度信息（如“已处理10/100条记录”）、自定义日志或指标。

In [19]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.config import get_stream_writer
from dotenv import load_dotenv
from rich import print as rprint
import os,time

load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

@tool
def generate_sales_report() -> str:
    """
    生成销售报告

    Returns:
        销售总结报告
    """
    writer = get_stream_writer()
    writer({"type": "生成销售报告", "message": "开始生产销售报告"})
    for i in range(1, 4):
        time.sleep(0.5)
        writer({"type": "生成销售报告", "message": f"生产销售报告进度百分比：{i * 25}%"})
    writer({"type": "生成销售报告", "message": "生产销售报告完成！"})
    return f"销售报告：总收入150万元，同比增长12%"

@tool
def generate_inventory_report() -> str:
    """
    生成库存报告

    Returns:
        库存报告
    """
    writer = get_stream_writer()
    writer("开始库存分析。。。")
    time.sleep(0.5)
    writer("检查当前库存量。。。")
    time.sleep(0.5)
    writer("生成库存报告。。。")
    return "当前库存量为10000件，库存不足，无异常"

# 2. 创建Agent
customer_service_agent = create_agent(
    model = model_openai,
    tools=[
        generate_sales_report,
        generate_inventory_report,
    ],
)

# 使用checkpointer模式进行流式监控
for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user", "content": "生成销售报告和库存报告"}]},
        stream_mode="custom"
):
    rprint(chunk)
    print("-" * 70)


{'type': '生成销售报告', 'message': '开始生产销售报告'}

----------------------------------------------------------------------


开始库存分析。。。

----------------------------------------------------------------------


{'type': '生成销售报告', 'message': '生产销售报告进度百分比：25%'}

...

----------------------------------------------------------------------


{'type': '生成销售报告', 'message': '生产销售报告进度百分比：75%'}

----------------------------------------------------------------------


{'type': '生成销售报告', 'message': '生产销售报告完成！'}

----------------------------------------------------------------------


| 模式 | 输出内容 | 使用场景|
| ----| ----| ---- |
|values|每个步骤执行后，都会输出完整的状态信息|适用于每一步都要获取完整状态、状态持久化场景|
|updates（默认）|每个步骤执行后，只增量更新状态中发生变化的内容|用于监控Agent 执行进度，例如观察Agent决定调用工具、工具执行结果等步骤|
|messages|输出流式返回的Token以及相关的元数据（如：来自哪个节点model/tool）|实现类似ChatGPT 的打字机效果，为聊天机器人等交互式应用提供最佳的实时体验|
|tasks|输出当前task任务开始和结束的时间,包含任务的结果和错误信息|该模式用于监控任务的生命周期|
|debug|与tasks模式类似，比task模式多输出任务步骤、时间戳、task类型（task/task_result）|该模式用于调试、监控task任务的生命周期|
|checkpoints|当检查点（checkpoint）被创建时会触发输出，输出包含检查点中的状态|用于需要状态持久化、工作流恢复或分布式执行跟踪的高级场景|
|custom|通过get_stream_writer在工具或节点内部自定义发送的数据|用于输出业务逻辑相关的进度信息（如“已处理10/100条记录”）、自定义日志或指标|

我们可以根据不同的目标来选择不同的输出模式。例如：
- 实现 实时对话交互 ，优先选择messages模式；
- 观察Agent的 思考与执行步骤 ，优先选择updates模式；
- 需要查看 每一步状态 优先选择values/tasks/debug模式；
- 在工具执行时 输出自定义业务 日志优先选择custom模式。

此外，以上这些模式还可以组合使用，例如，可以同时指定 stream_mode=[“tasks”，“updates”] ，这样在同一个循环里 既能 查看Agent task任务执行内容， 又能 显示Agent每步的更新。

In [20]:
from typing import Dict, Any
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
import os

load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID,用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三", "level": "VIP", "join_date": "2022-12-02"}

def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID,用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费数等
    """
    return {"total_orders": 14, "total_spent": 287192.83}

@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-07-07"
    }

customer_service_agent = create_agent(
    model = model_openai,
    tools=[
        query_customer_data,
        check_order_history,
        get_current_promotions,
    ]
)

for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user", "content": "查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠"}]},
        stream_mode=["updates","tasks"]
):
    rprint(chunk)
    print("-" * 70)

(
    'tasks',
    {
        'id': '546279e6-e6b4-7639-7ea2-4acd911465c0',
        'name': 'model',
        'input': {
            'messages': [
                HumanMessage(
                    content='查询客户ID为CUSTID1234567的个人信息、历史订单和可优惠',
                    additional_kwargs={},
                    response_metadata={},
                    id='8e6acf5c-57f9-4909-93dd-0e6a96269771'
                )
            ]
        },
        'triggers': ('branch:to:model',),
        'metadata': {'ls_integration': 'langchain_create_agent'}
    }
)

----------------------------------------------------------------------


(
    'updates',
    {
        'model': {
            'messages': [
                AIMessage(
                    content='',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 693,
                            'prompt_tokens': 488,
                            'total_tokens': 1181,
                            'completion_tokens_details': {
                                'accepted_prediction_tokens': None,
                                'audio_tokens': None,
                                'reasoning_tokens': 600,
                                'rejected_prediction_tokens': None,
                                'text_tokens': 693
                            },
                            'prompt_tokens_details': {
                                'audio_tokens': None,
                                'cache_write_tokens': None,
                                'cached_tokens': 0,
                                'text_tokens': 488
                            }
                        },
                        'model_provider': 'openai',
                        'model_name': 'qwen3.7-flash',
                        'system_fingerprint': None,
                        'id': 'chatcmpl-9ef9a4d1-8e9b-9a86-ab49-fa6a519d1207',
                        'finish_reason': 'tool_calls',
                        'logprobs': None
                    },
                    id='lc_run--01a0c429-f73a-7232-b146-5a4dd53e9cc2-0',
                    tool_calls=[
                        {
                            'name': 'query_customer_data',
                            'args': {'customer_id': 'CUSTID1234567'},
                            'id': 'call_151a503c7d2944cf8fe95ec2',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'check_order_history',
                            'args': {'customer_id': 'CUSTID1234567'},
                            'id': 'call_7ee89b13e7df47e485c147e8',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'get_current_promotions',
                            'args': {},
                            'id': 'call_d544c98a1f314946bc5af4ef',
                            'type': 'tool_call'
                        }
                    ],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 488,
                        'output_tokens': 693,
                        'total_tokens': 1181,
                        'input_token_details': {'cache_read': 0},
                        'output_token_details': {'reasoning': 600}
                    }
                )
            ]
        }
    }
)

----------------------------------------------------------------------


(
    'tasks',
    {
        'id': '546279e6-e6b4-7639-7ea2-4acd911465c0',
        'name': 'model',
        'error': None,
        'result': {
            'messages': [
                AIMessage(
                    content='',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 693,
                            'prompt_tokens': 488,
                            'total_tokens': 1181,
                            'completion_tokens_details': {
                                'accepted_prediction_tokens': None,
                                'audio_tokens': None,
                                'reasoning_tokens': 600,
                                'rejected_prediction_tokens': None,
                                'text_tokens': 693
                            },
                            'prompt_tokens_details': {
                                'audio_tokens': None,
                                'cache_write_tokens': None,
                                'cached_tokens': 0,
                                'text_tokens': 488
                            }
                        },
                        'model_provider': 'openai',
                        'model_name': 'qwen3.7-flash',
                        'system_fingerprint': None,
                        'id': 'chatcmpl-9ef9a4d1-8e9b-9a86-ab49-fa6a519d1207',
                        'finish_reason': 'tool_calls',
                        'logprobs': None
                    },
                    id='lc_run--01a0c429-f73a-7232-b146-5a4dd53e9cc2-0',
                    tool_calls=[
                        {
                            'name': 'query_customer_data',
                            'args': {'customer_id': 'CUSTID1234567'},
                            'id': 'call_151a503c7d2944cf8fe95ec2',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'check_order_history',
                            'args': {'customer_id': 'CUSTID1234567'},
                            'id': 'call_7ee89b13e7df47e485c147e8',
                            'type': 'tool_call'
                        },
                        {
                            'name': 'get_current_promotions',
                            'args': {},
                            'id': 'call_d544c98a1f314946bc5af4ef',
                            'type': 'tool_call'
                        }
                    ],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 488,
                        'output_tokens': 693,
                        'total_tokens': 1181,
                        'input_token_details': {'cache_read': 0},
                        'output_token_details': {'reasoning': 600}
                    }
                )
            ]
        },
        'interrupts': []
    }
)

...

----------------------------------------------------------------------


(
    'updates',
    {
        'model': {
            'messages': [
                AIMessage(
                    content='根据您提供的信息，以下是关于客户ID CUSTID1234567的相关数据：\n\n**个人信息：**\n* 
姓名：张三\n* 等级：VIP\n* 加入日期：2022-12-02\n\n**历史订单：**\n* 总订单数：14\n* 
总花费金额：¥287,192.83\n\n**当前可用优惠活动：**\n* 活动名称：老用户优惠、会员专属折扣\n* 有效期至：2027-07-07',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 173,
                            'prompt_tokens': 680,
                            'total_tokens': 853,
                            'completion_tokens_details': {
                                'accepted_prediction_tokens': None,
                                'audio_tokens': None,
                                'reasoning_tokens': 50,
                                'rejected_prediction_tokens': None,
                                'text_tokens': 173
                            },
                            'prompt_tokens_details': {
                                'audio_tokens': None,
                                'cache_write_tokens': None,
                                'cached_tokens': 0,
                                'text_tokens': 680
                            }
                        },
                        'model_provider': 'openai',
                        'model_name': 'qwen3.7-flash',
                        'system_fingerprint': None,
                        'id': 'chatcmpl-5ac76f7e-9ff5-96fd-a46f-035233c3a315',
                        'finish_reason': 'stop',
                        'logprobs': None
                    },
                    id='lc_run--01a0c42a-1b98-75f0-808a-d5b93e19a8c7-0',
                    tool_calls=[],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 680,
                        'output_tokens': 173,
                        'total_tokens': 853,
                        'input_token_details': {'cache_read': 0},
                        'output_token_details': {'reasoning': 50}
                    }
                )
            ]
        }
    }
)

----------------------------------------------------------------------


(
    'tasks',
    {
        'id': '4c52d947-b8d1-e8b7-1a23-0542fc552345',
        'name': 'model',
        'error': None,
        'result': {
            'messages': [
                AIMessage(
                    content='根据您提供的信息，以下是关于客户ID CUSTID1234567的相关数据：\n\n**个人信息：**\n* 
姓名：张三\n* 等级：VIP\n* 加入日期：2022-12-02\n\n**历史订单：**\n* 总订单数：14\n* 
总花费金额：¥287,192.83\n\n**当前可用优惠活动：**\n* 活动名称：老用户优惠、会员专属折扣\n* 有效期至：2027-07-07',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 173,
                            'prompt_tokens': 680,
                            'total_tokens': 853,
                            'completion_tokens_details': {
                                'accepted_prediction_tokens': None,
                                'audio_tokens': None,
                                'reasoning_tokens': 50,
                                'rejected_prediction_tokens': None,
                                'text_tokens': 173
                            },
                            'prompt_tokens_details': {
                                'audio_tokens': None,
                                'cache_write_tokens': None,
                                'cached_tokens': 0,
                                'text_tokens': 680
                            }
                        },
                        'model_provider': 'openai',
                        'model_name': 'qwen3.7-flash',
                        'system_fingerprint': None,
                        'id': 'chatcmpl-5ac76f7e-9ff5-96fd-a46f-035233c3a315',
                        'finish_reason': 'stop',
                        'logprobs': None
                    },
                    id='lc_run--01a0c42a-1b98-75f0-808a-d5b93e19a8c7-0',
                    tool_calls=[],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 680,
                        'output_tokens': 173,
                        'total_tokens': 853,
                        'input_token_details': {'cache_read': 0},
                        'output_token_details': {'reasoning': 50}
                    }
                )
            ]
        },
        'interrupts': []
    }
)

----------------------------------------------------------------------
